# Lab: Clinical Decision Support — Heart Disease Referral Alerting

**Syllabus tie-in:** Unit II — AI Applications Across the Healthcare Ecosystem — *Clinical Decision Support Systems*

**Objective:** Practice, on your own, the same steps you used in the Week 2 diabetes lab — exploratory data analysis, data cleaning, normalization, model training, and evaluation — this time on a new dataset with a new twist: encoding categorical clinical codes, and treating the **alert threshold as a clinical decision**, not a modeling default.

**How this notebook works:** This is a **practice lab, not a worked tutorial**. Each section explains a concept and tells you what to produce; the code cells are for you to fill in. Refer back to the diabetes notebook and lecture material for the *how* — this lab is where you apply it yourself. A separate solution notebook exists for you to check your work against **after** you've made a genuine attempt — don't open it first.

**Important framing before we start:** This model is a **teaching exercise on a public research dataset**, not a validated clinical tool. Nothing it outputs should be read as a diagnosis. We are practicing the *mechanics* of building a decision-support alert — the much harder problem of proving one is safe to deploy is Unit IV/V territory.


Run the cell below as-is — it just loads the libraries you'll need. You used all of these in the diabetes lab except `precision_recall_curve`, which you'll use in Section 5.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve, roc_auc_score,
    precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid')


## 1. The Clinical Scenario and the Dataset

**Scenario:** A general physician sees patients for routine checkups. Some non-invasive tests are already collected (age, blood pressure, cholesterol, resting ECG, an exercise stress test). The clinic wants a simple tool that flags which patients should be referred to a cardiologist for further (invasive) evaluation — rather than referring everyone, or relying only on physician judgement, which varies.

**Dataset:** UCI **Heart Disease (Cleveland)** dataset — 303 patients, 13 clinical attributes, collected at the Cleveland Clinic Foundation. This is one of the most widely used datasets in clinical ML research.

| Column | Meaning |
|---|---|
| age | Age in years |
| sex | 1 = male, 0 = female |
| cp | Chest pain type (1: typical angina, 2: atypical angina, 3: non-anginal pain, 4: asymptomatic) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) |
| restecg | Resting ECG results (0: normal, 1: ST-T abnormality, 2: probable/definite left ventricular hypertrophy) |
| thalach | Maximum heart rate achieved (exercise stress test) |
| exang | Exercise-induced angina (1 = yes, 0 = no) |
| oldpeak | ST depression induced by exercise relative to rest |
| slope | Slope of the peak exercise ST segment (1: upsloping, 2: flat, 3: downsloping) |
| ca | Number of major vessels (0–3) colored by fluoroscopy |
| thal | 3 = normal, 6 = fixed defect, 7 = reversible defect |
| target | Angiographic disease status: 0 = no disease, 1–4 = disease present (increasing severity) |

We'll binarize `target` into "disease present" (1) vs. "no disease" (0), which is the standard approach for this dataset — the severity grades (1–4) are not reliably distinguishable from these 13 features alone.


In [ ]:
url = "https://raw.githubusercontent.com/AndreasPr/Heart-Disease-Prediction-System/master/processed.cleveland.data.csv"

columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

# '?' marks missing values in the original UCI file
df = pd.read_csv(url, names=columns, na_values="?")

# Binarize target: 0 = no disease, 1 = disease present (any severity)
df["target"] = (df["target"] > 0).astype(int)

print("First 5 rows:")
df.head()


In [ ]:
print(f"Number of patients: {len(df)}")
print(f"Number of features: {df.shape[1] - 1}")


## 2. Exploratory Data Analysis

### 2.1. Missing Values

Unlike the diabetes dataset, missingness here is **explicit** (`?` in the source file, already converted to `NaN` when we loaded the data) rather than disguised as biologically-impossible zeros.

**Your task:**
1. Print how many missing values each column has.
2. You should find missing values in exactly two columns. Both are effectively categorical/ordinal with only a handful of possible values — think about why the *mean* (your default from the diabetes lab) would produce a nonsensical value here, and impute with the **mode** instead.


In [ ]:
# TODO: print the number of missing values per column



In [ ]:
# TODO: impute the missing values in the two affected columns using each column's mode
# Hint: df[col].mode()[0] gives the most frequent value of a column

# TODO: re-check that there are no missing values left



### 2.2. Encoding Categorical Features

This is new compared to the diabetes lab: several columns here are **categorical codes**, not continuous measurements, even though they're stored as numbers:

- `cp`, `restecg`, `slope`, `thal` are **nominal** (the numeric codes have no inherent order) — these need one-hot encoding, or a model like logistic regression will wrongly treat e.g. `thal=7` as "more" of something than `thal=3`.
- `sex`, `fbs`, `exang` are already binary (0/1) — no change needed.
- `ca` is a genuine count (0–3 vessels) — leave it numeric since more vessels affected is meaningfully "more."

**Your task:**
1. One-hot encode the four nominal columns listed above (`pd.get_dummies` is your friend, same as you may have used for categorical features before — check the pandas documentation if this is new to you).
2. Store the result in a new dataframe, e.g. `df_encoded`.
3. **Tip:** the nominal columns loaded as decimals (e.g. `3.0` not `3`). If you one-hot encode them as-is, you'll get messy column names like `cp_2.0`. Consider converting those columns to integer type *before* encoding.


In [ ]:
# TODO: cast the categorical/binary columns to int (to avoid messy "2.0"-style column names)

# TODO: one-hot encode the nominal columns (cp, restecg, slope, thal) into df_encoded

# TODO: print df.shape[1] and df_encoded.shape[1] to confirm new columns were added



### 2.3. Target Distribution

Worth comparing to the diabetes lab: that dataset was noticeably imbalanced (~65% / 35%).

**Your task:** Plot the distribution of `target` in `df_encoded` (a count plot works well, same as the diabetes lab) and print the class proportions. Is this dataset more or less balanced than the diabetes one? Note your answer — it matters for how you interpret metrics later.


In [ ]:
# TODO: plot the target distribution and print class proportions



### 2.4. Feature Distributions

**Your task:** Plot the distribution (histogram, with a KDE overlay if you like) of the continuous features: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`. A grid of subplots (as in the diabetes lab) is a clean way to show all five at once.


In [ ]:
# TODO: plot histograms of the continuous features listed above



### 2.5. Outliers

**Your task:** Produce boxplots of the same continuous features to check for outliers. As in the diabetes lab, don't automatically drop anything — a few outliers may be clinically genuine (e.g. very high cholesterol). Just note where they are.


In [ ]:
# TODO: plot boxplots of the continuous features



### 2.6. Correlation Matrix

**Your task:** Plot a correlation heatmap of `df_encoded` (all columns, post-encoding). Which features correlate most strongly with `target`?


In [ ]:
# TODO: plot a correlation heatmap of df_encoded



## 3. Data Preparation for Modeling

**Your task:**
1. Split `df_encoded` into `X` (all columns except `target`) and `y` (`target`).
2. Split `X` and `y` into training and test sets (80/20), stratified on `y`, same random_state convention as the diabetes lab (`random_state=42`) so results are reproducible.
3. Scale the features with `StandardScaler` — remember: **fit only on the training data**, then transform both train and test, to avoid leaking test-set information.


In [ ]:
# TODO: create X and y from df_encoded

# TODO: train_test_split into X_train, X_test, y_train, y_test (80/20, stratified, random_state=42)



In [ ]:
# TODO: fit a StandardScaler on X_train only, then transform both X_train and X_test
# Store the results as X_train_scaled and X_test_scaled



## 4. Model Training and Standard Evaluation

Same model family as the diabetes lab — logistic regression.

**Your task:**
1. Train a `LogisticRegression` model on the scaled training data.
2. Print the standardized coefficients alongside their feature names, sorted, so you can see which features matter most (same as the diabetes lab's feature-importance step).


In [ ]:
# TODO: train a LogisticRegression model on X_train_scaled, y_train

# TODO: build and print a dataframe of feature names + coefficients, sorted by coefficient value



**Your task:** Using the default 0.5 threshold:
1. Compute predictions (`y_pred`) and predicted probabilities (`y_pred_proba`) on the test set.
2. Print accuracy.
3. Plot a confusion matrix.
4. Print a full classification report.
5. Plot the ROC curve and print the AUC score.


In [ ]:
# TODO: compute y_pred and y_pred_proba on the test set

# TODO: print accuracy



In [ ]:
# TODO: compute and plot the confusion matrix



In [ ]:
# TODO: print the classification report



In [ ]:
# TODO: compute AUC, plot the ROC curve (include a diagonal reference line for random guessing)



## 5. From "Model" to "Clinical Decision Support Tool": Choosing the Alert Threshold

Everything above used the default 0.5 cutoff: if the model's estimated probability of disease is above 50%, predict "disease." **That 0.5 is a modeling default, not a clinical decision.** A real CDS tool's threshold should reflect the relative cost of the two kinds of mistakes:

- **False Negative** (model says "no disease," patient actually has it): a missed referral — the patient doesn't see a cardiologist when they should.
- **False Positive** (model says "disease," patient doesn't have it): an unnecessary referral — costs clinician time, patient anxiety, and healthcare-system resources, but is far less dangerous than a miss.

In most screening contexts, missing a case is considered worse than an unnecessary referral — which argues for a **lower** threshold than 0.5 (more sensitive, more referrals, catches more true cases, but also more false alarms). Let's see this trade-off directly instead of just asserting it.

**Your task:** Using `precision_recall_curve(y_test, y_pred_proba)`, plot precision and recall against threshold, and also plot the **referral rate** (the fraction of test patients flagged) at each threshold — `(y_pred_proba >= t).mean()` for each threshold `t`.


In [ ]:
# TODO: compute precisions, recalls, pr_thresholds using precision_recall_curve

# TODO: compute referral_rates at each threshold

# TODO: plot precision, recall, and referral rate all against threshold on one chart
# (a twin y-axis, as shown in lecture, works well since referral rate and precision/recall
#  are on different scales)



### Your task

Your clinic estimates that **missing a heart-disease case costs roughly 5 times as much** (in downstream complications, worse outcomes, potential liability) **as one unnecessary cardiology referral** (clinician time, patient inconvenience).

Using the plot above:

1. Pick a threshold that reflects this 5:1 cost ratio — you don't need a precise optimization, a reasoned choice from the chart is fine (hint: a threshold that favors recall over precision, since misses are costlier).
2. Recompute the confusion matrix and classification report at your chosen threshold using the cell below.
3. In a markdown cell, write 3–4 sentences: what threshold did you pick, what happened to sensitivity and the referral rate compared to the default, and what would you tell the clinic about the trade-off you made?


In [ ]:
# TODO: set your chosen threshold based on the cost ratio discussed above
my_threshold = None  # replace with a number between 0 and 1

# TODO: recompute predictions at your threshold and print the confusion matrix + classification report



*(Your written justification goes here — replace this text.)*


## 6. Seeing the Alert in Action: Three Patients at the Clinic

Let's simulate three patients arriving for a routine checkup and see what the CDS tool would say — first at the default 0.5 threshold, then at your chosen threshold from Section 5.

The cell below already defines three patient profiles for you (a data-entry task isn't the point of this exercise) — column order matches `X.columns` automatically.

**Your task:** Scale the sample patients with your already-fitted scaler, get predicted probabilities from your trained model, and build a small results table showing each patient's probability and alert decision at (a) the default 0.5 threshold and (b) your chosen threshold from Section 5.


In [ ]:
sample_patients = pd.DataFrame([
    {  # Profile A: several risk factors present
        "age": 62, "sex": 1, "trestbps": 152, "chol": 275, "fbs": 0,
        "thalach": 118, "exang": 1, "oldpeak": 2.4, "ca": 2,
        "cp_2": 0, "cp_3": 0, "cp_4": 1,
        "restecg_1": 0, "restecg_2": 1,
        "slope_2": 1, "slope_3": 0,
        "thal_6": 0, "thal_7": 1,
    },
    {  # Profile B: mostly low risk factors
        "age": 41, "sex": 0, "trestbps": 118, "chol": 190, "fbs": 0,
        "thalach": 172, "exang": 0, "oldpeak": 0.2, "ca": 0,
        "cp_2": 1, "cp_3": 0, "cp_4": 0,
        "restecg_1": 0, "restecg_2": 0,
        "slope_2": 0, "slope_3": 0,
        "thal_6": 0, "thal_7": 0,
    },
    {  # Profile C: borderline / mixed signals
        "age": 55, "sex": 1, "trestbps": 130, "chol": 233, "fbs": 1,
        "thalach": 150, "exang": 0, "oldpeak": 1.0, "ca": 0,
        "cp_2": 0, "cp_3": 1, "cp_4": 0,
        "restecg_1": 0, "restecg_2": 0,
        "slope_2": 1, "slope_3": 0,
        "thal_6": 0, "thal_7": 0,
    },
])[X.columns]  # ensure column order matches training data


In [ ]:
# TODO: scale sample_patients using your already-fitted scaler

# TODO: get predicted probabilities for each sample patient

# TODO: build a results table with columns: Patient, Predicted Probability,
#       Alert at 0.5 (default), Alert at your chosen threshold



Check: does any patient's alert *change* between the default threshold and your chosen one? If so, that's exactly the kind of patient where the threshold decision matters most in practice.


## 7. Discussion Questions (for lab writeup or in-class discussion)

These connect this lab to next week's Unit III material on risk and human-centered AI — no code needed, just written responses.

1. If you lower the alert threshold to catch more true cases, referrals go up. At what point would you worry about **alert fatigue** — clinicians starting to ignore the tool because it fires too often?
2. Who should be responsible for deciding this threshold: the AI developer, the hospital administration, the treating clinician, or a clinical guidelines committee? Why?
3. This model was trained on 303 patients from one hospital in Cleveland in the 1980s. What could go wrong if it's deployed, unmodified, at a hospital in a different country with a different patient population — this is a preview of *dataset shift*, coming up in Unit III.
4. The interface only needs to show "refer" or "don't refer." What else should a clinician see alongside that alert (e.g., the probability, the top contributing factors, a confidence indicator) to trust and use it well?


## 8. Conclusion & Next Steps

By completing this lab yourself, you've practiced:

1. Loading and cleaning real clinical data with genuinely missing values (not disguised as zeros).
2. Encoding categorical clinical codes correctly instead of treating them as ordinary numbers.
3. Training and evaluating a logistic regression model the same way as last week — but this time without a worked example to copy from.
4. Treating the **alert threshold as a clinical policy decision**, tied to the relative cost of a missed diagnosis vs. an unnecessary referral — not a modeling default.
5. Applying your trained model to individual patients, and connecting the results to human-centered questions (alert fatigue, accountability, dataset shift) that the course returns to shortly.

**Next lab:** explainability (SHAP / feature contributions) and calibration — so that instead of just an alert, the tool can also say *why* it's flagging a patient, and how much to trust its stated probability.
